# 11. Auditoria de informacion minima para el flujo LLM v3

Este notebook estudia cuanta informacion inicial necesita el flujo de produccion para mantener una extraccion y una prediccion razonablemente estables. No entrena modelos, no recalibra probabilidades y no modifica la politica A1.

La comparacion se hace por ablacion: para cada historia se compara la narrativa completa frente a versiones mas cortas con menos campos disponibles.

## 1. Configuracion

Se usa el extractor de produccion, que debe cargar `prompts/extractor_system_v3_final.txt` desde `PROMPTS_DIR`.

In [1]:
import hashlib
import time

import numpy as np
import pandas as pd

from triaje_ia.config import PROMPTS_DIR
from triaje_ia.inference.predictor import TriajePredictor
from triaje_ia.llm.extractor import SYSTEM_PROMPT, extraer_vector_clinico
from triaje_ia.llm.validator import resumen_validacion, validar_vector_clinico

PROMPT_PATH = PROMPTS_DIR / "extractor_system_v3_final.txt"
LLM_MODEL = "llama3.1:8b-instruct-q4_K_M"

assert PROMPT_PATH.exists(), f"No existe el prompt esperado: {PROMPT_PATH}"
assert SYSTEM_PROMPT == PROMPT_PATH.read_text(encoding="utf-8")

predictor = TriajePredictor()
pd.DataFrame([
    {"elemento": "prompt", "valor": PROMPT_PATH.name},
    {"elemento": "prompt_sha256", "valor": hashlib.sha256(SYSTEM_PROMPT.encode("utf-8")).hexdigest()},
    {"elemento": "modelo_llm", "valor": LLM_MODEL},
    {"elemento": "politica_decision", "valor": predictor._decision_policy},
    {"elemento": "umbral_alerta_a1", "valor": predictor._warning_threshold_a1},
])

c:\Users\CARLOS\triaje-ia-tfg\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,elemento,valor
0,prompt,extractor_system_v3_final.txt
1,prompt_sha256,aa834ac9762fcd0d0fda525a9b20943edc9718226c059e...
2,modelo_llm,llama3.1:8b-instruct-q4_K_M
3,politica_decision,argmax_with_a1_warning
4,umbral_alerta_a1,0.4


## 2. Historias y variantes de informacion

Cada caso tiene cinco versiones. La version completa se toma como referencia interna de esta auditoria. No se asume que sea una verdad clinica perfecta; solo sirve para medir sensibilidad al texto disponible.

In [2]:
CASOS = [
    {"id": "S01", "categoria": "critico_neuro", "motivo": "Bajo nivel de consciencia brusco hace 1h.", "edad_sexo": "Varon 67a. Bajo nivel de consciencia brusco hace 1h.", "motivo_constantes": "Varon 67a. Bajo nivel de consciencia brusco hace 1h. TA 185/110, FC 48 irregular, FR 14, SatO2 94%, T 36.2.", "motivo_antecedentes": "Varon 67a. HTA y FA cronica. Tto bisoprolol y acenocumarol. Bajo nivel de consciencia brusco hace 1h.", "completa": "Varon 67a. HTA y FA cronica. Tto bisoprolol y acenocumarol. Traido por familia por bajo nivel de consciencia brusco hace 1h. Sin fiebre. TA 185/110, FC 48 irregular, FR 14, SatO2 94%, T 36.2. GCS 10."},
    {"id": "S02", "categoria": "dolor_toracico", "motivo": "Dolor toracico opresivo con sudoracion y nauseas.", "edad_sexo": "Mujer 72a. Dolor toracico opresivo de 45 min con sudoracion y nauseas.", "motivo_constantes": "Mujer 72a. Dolor toracico opresivo de 45 min. TA 92/58, FC 118, FR 24, SatO2 91%, T 36.8. Dolor 9/10.", "motivo_antecedentes": "Mujer 72a con diabetes e hipertension. Dolor toracico opresivo de 45 min con sudoracion y nauseas.", "completa": "Mujer 72a con diabetes e hipertension. Dolor toracico opresivo de 45 min con sudoracion y nauseas. TA 92/58, FC 118, FR 24, SatO2 91%, T 36.8. Dolor 9/10."},
    {"id": "S03", "categoria": "disnea", "motivo": "Disnea progresiva desde ayer y tos.", "edad_sexo": "Varon 64a. Disnea progresiva desde ayer y tos.", "motivo_constantes": "Varon 64a. Disnea progresiva desde ayer y tos. TA 145/82, FC 108, FR 30, SatO2 86%, T 37.9.", "motivo_antecedentes": "Varon 64a EPOC. Disnea progresiva desde ayer y tos.", "completa": "Varon 64a EPOC. Consulta por disnea progresiva desde ayer y tos. TA 145/82, FC 108, FR 30, SatO2 86%, T 37.9. No dolor toracico."},
    {"id": "S04", "categoria": "sepsis", "motivo": "Fiebre y confusion desde esta manana.", "edad_sexo": "Mujer 81a. Fiebre y confusion desde esta manana.", "motivo_constantes": "Mujer 81a. Fiebre y confusion. TA 88/52, FC 124, FR 28, SatO2 92%, T 39.1.", "motivo_antecedentes": "Mujer 81a con insuficiencia renal. Fiebre y deterioro general desde hace 2 dias, confusion desde esta manana.", "completa": "Mujer 81a con insuficiencia renal. Fiebre y deterioro general desde hace 2 dias, confusion desde esta manana. TA 88/52, FC 124, FR 28, SatO2 92%, T 39.1."},
    {"id": "S05", "categoria": "cefalea", "motivo": "Cefalea brusca, la peor de su vida, con vomitos y fotofobia.", "edad_sexo": "Hombre de unos 40 anos con cefalea brusca, la peor de su vida, desde hace 30 min.", "motivo_constantes": "Hombre 40a. Cefalea brusca, vomitos y fotofobia. TA 160/95, FC 96, FR 18, SatO2 98%, T 36.7.", "motivo_antecedentes": "Hombre 40a sin antecedentes referidos. Cefalea brusca, vomitos y fotofobia.", "completa": "Hombre de unos 40 anos con cefalea brusca, la peor de su vida, desde hace 30 min. Vomitos y fotofobia. TA 160/95, FC 96, FR 18, SatO2 98%, T 36.7."},
    {"id": "S06", "categoria": "abdomen", "motivo": "Dolor hipocondrio derecho y nauseas.", "edad_sexo": "Mujer 52a dolor hipocondrio derecho desde la cena, nauseas.", "motivo_constantes": "Mujer 52a dolor hipocondrio derecho desde la cena, nauseas. TA 134/78, FC 92, FR 18, SatO2 98%, T 37.4. EVA 6/10.", "motivo_antecedentes": "Mujer 52a sin antecedentes relevantes mencionados. Dolor hipocondrio derecho desde la cena, nauseas.", "completa": "Mujer 52a dolor hipocondrio derecho desde la cena, nauseas. TA 134/78, FC 92, FR 18, SatO2 98%, T 37.4, Glasgow 15. EVA 6/10."},
    {"id": "S07", "categoria": "abdomen_fid", "motivo": "Dolor en fosa iliaca derecha y vomitos.", "edad_sexo": "Varon 29a dolor en fosa iliaca derecha de 10 horas de evolucion, vomitos.", "motivo_constantes": "Varon 29a dolor FID 10 horas, vomitos. TA 122/74, FC 101, FR 18, SatO2 99%, T 38.0. Dolor 7/10.", "motivo_antecedentes": "Varon 29a sin antecedentes mencionados. Dolor FID de 10 horas y vomitos.", "completa": "Varon 29a dolor en fosa iliaca derecha de 10 horas de evolucion, vomitos en dos ocasiones. TA 122/74, FC 101, FR 18, SatO2 99%, T 38.0. Dolor 7/10."},
    {"id": "S08", "categoria": "sincope", "motivo": "Sincope en domicilio con recuperacion completa.", "edad_sexo": "Mujer 58a sincope en domicilio con recuperacion completa.", "motivo_constantes": "Mujer 58a sincope con recuperacion completa. TA 110/70, FC 54, FR 16, SatO2 97%, T 36.4.", "motivo_antecedentes": "Mujer 58a con HTA. Sincope en domicilio con recuperacion completa. Niega dolor toracico.", "completa": "Mujer 58a sincope en domicilio con recuperacion completa. Mareo previo. HTA. TA 110/70, FC 54, FR 16, SatO2 97%, T 36.4. Niega dolor toracico."},
    {"id": "S09", "categoria": "trauma", "motivo": "Caida de bicicleta, dolor intenso en muneca derecha.", "edad_sexo": "Varon 35a caida de bicicleta hace 1h, dolor intenso en muneca derecha.", "motivo_constantes": "Varon 35a caida bicicleta 1h, dolor muneca derecha. TA 128/80, FC 88, FR 16, SatO2 99%, T 36.6. Dolor 8/10.", "motivo_antecedentes": "Varon 35a sin antecedentes mencionados. Caida de bicicleta, dolor en muneca y herida superficial en rodilla.", "completa": "Varon 35a caida de bicicleta hace 1h, dolor intenso en muneca derecha y herida superficial en rodilla. TA 128/80, FC 88, FR 16, SatO2 99%, T 36.6. Dolor 8/10."},
    {"id": "S10", "categoria": "alergia", "motivo": "Urticaria tras frutos secos y sensacion de garganta cerrada.", "edad_sexo": "Mujer 24a urticaria generalizada tras comer frutos secos, sensacion de garganta cerrada.", "motivo_constantes": "Mujer 24a urticaria y garganta cerrada tras frutos secos. TA 100/65, FC 120, FR 26, SatO2 95%, T 36.5.", "motivo_antecedentes": "Mujer 24a sin antecedentes mencionados. Urticaria tras frutos secos y sensacion de garganta cerrada.", "completa": "Mujer 24a urticaria generalizada tras comer frutos secos, sensacion de garganta cerrada. TA 100/65, FC 120, FR 26, SatO2 95%, T 36.5."},
    {"id": "S11", "categoria": "psiquiatrico", "motivo": "Ideacion autolitica y ansiedad intensa.", "edad_sexo": "Varon 46a traido por familia por ideacion autolitica.", "motivo_constantes": "Varon 46a ideacion autolitica, ansiedad intensa. TA 136/84, FC 112, FR 20, SatO2 98%, T 36.7.", "motivo_antecedentes": "Varon 46a traido por familia por ideacion autolitica. Ansiedad intensa, sin lesiones.", "completa": "Varon 46a traido por familia por ideacion autolitica. Ansiedad intensa, sin lesiones. TA 136/84, FC 112, FR 20, SatO2 98%, T 36.7."},
    {"id": "S12", "categoria": "rectorragia", "motivo": "Rectorragia desde ayer y debilidad.", "edad_sexo": "Mujer 69a rectorragia desde ayer y debilidad.", "motivo_constantes": "Mujer 69a rectorragia y debilidad. TA 98/60, FC 116, FR 20, SatO2 97%, T 36.3.", "motivo_antecedentes": "Mujer 69a anticoagulada con acenocumarol por FA. Rectorragia desde ayer y debilidad.", "completa": "Mujer 69a anticoagulada con acenocumarol por FA. Rectorragia desde ayer y debilidad. TA 98/60, FC 116, FR 20, SatO2 97%, T 36.3."},
    {"id": "S13", "categoria": "convulsion", "motivo": "Convulsion tonico-clonica hace 20 min, ahora somnoliento.", "edad_sexo": "Varon 31a convulsion tonico-clonica hace 20 min, ahora somnoliento.", "motivo_constantes": "Varon 31a convulsion hace 20 min, somnoliento. TA 130/82, FC 105, FR 18, SatO2 96%, T 36.8.", "motivo_antecedentes": "Varon 31a epilepsia conocida. Convulsion tonico-clonica hace 20 min, ahora somnoliento.", "completa": "Varon 31a epilepsia conocida, convulsion tonico-clonica hace 20 min, ahora somnoliento. TA 130/82, FC 105, FR 18, SatO2 96%, T 36.8."},
    {"id": "S14", "categoria": "fiebre_leve", "motivo": "Fiebre, tos y mialgias desde ayer.", "edad_sexo": "Mujer 36a fiebre, tos y mialgias desde ayer.", "motivo_constantes": "Mujer 36a fiebre, tos y mialgias. TA 118/72, FC 98, FR 18, SatO2 98%, T 38.4. Dolor 3/10.", "motivo_antecedentes": "Mujer 36a sin antecedentes mencionados. Fiebre, tos y mialgias desde ayer. Sin disnea.", "completa": "Mujer 36a fiebre, tos y mialgias desde ayer. Sin disnea. TA 118/72, FC 98, FR 18, SatO2 98%, T 38.4. Dolor 3/10."},
    {"id": "S15", "categoria": "otalgia", "motivo": "Dolor de oido derecho desde ayer.", "edad_sexo": "Mujer 34a dolor de oido derecho desde ayer, febricula.", "motivo_constantes": "Mujer 34a dolor de oido derecho desde ayer. TA 122/70, FC 84, FR 16, SatO2 99%, T 37.5.", "motivo_antecedentes": "Mujer 34a dolor de oido derecho desde ayer, febricula. Sin dolor dental.", "completa": "Mujer 34a dolor de oido derecho desde ayer, febricula. Sin dolor dental. TA 122/70, FC 84, FR 16, SatO2 99%, T 37.5."},
    {"id": "S16", "categoria": "dental", "motivo": "Dolor dental en molar inferior desde hace 3 dias.", "edad_sexo": "Varon 45a dolor dental en molar inferior desde hace 3 dias.", "motivo_constantes": "Varon 45a dolor dental 3 dias. TA 130/80, FC 86, FR 16, SatO2 99%, T 36.7. Dolor 6/10.", "motivo_antecedentes": "Varon 45a dolor dental en molar inferior desde hace 3 dias. No fiebre.", "completa": "Varon 45a dolor dental en molar inferior desde hace 3 dias. No fiebre. TA 130/80, FC 86, FR 16, SatO2 99%, T 36.7. Dolor 6/10."},
    {"id": "S17", "categoria": "una_encarnada", "motivo": "Dolor en dedo gordo del pie por una una encarnada.", "edad_sexo": "Varon 41a dolor en dedo gordo del pie por una una encarnada.", "motivo_constantes": "Varon 41a dolor dedo gordo pie por una encarnada. TA 128/76, FC 82, FR 16, SatO2 98%, T 36.7.", "motivo_antecedentes": "Varon 41a dolor en dedo gordo del pie por una encarnada, rojo y algo inflamado. No fiebre.", "completa": "Varon 41a dolor en dedo gordo del pie por una una encarnada, rojo y algo inflamado. No fiebre. TA 128/76, FC 82, FR 16, SatO2 98%, T 36.7."},
    {"id": "S18", "categoria": "cura", "motivo": "Revision de herida quirurgica limpia y cura programada.", "edad_sexo": "Mujer 57a acude para revision de herida quirurgica limpia.", "motivo_constantes": "Mujer 57a revision herida quirurgica limpia. TA 126/78, FC 76, FR 16, SatO2 99%, T 36.4.", "motivo_antecedentes": "Mujer 57a revision de herida quirurgica limpia. Niega dolor y fiebre.", "completa": "Mujer 57a acude para revision de herida quirurgica limpia y cura programada. Niega dolor y fiebre. TA 126/78, FC 76, FR 16, SatO2 99%, T 36.4."},
    {"id": "S19", "categoria": "receta", "motivo": "Solicita receta de medicacion habitual porque se le acabo.", "edad_sexo": "Varon 55a solicita receta de medicacion habitual porque se le acabo.", "motivo_constantes": "Varon 55a solicita receta de medicacion habitual. TA 130/76, FC 74, FR 16, SatO2 98%, T 36.5.", "motivo_antecedentes": "Varon 55a solicita receta de medicacion habitual porque se le acabo. Niega sintomas.", "completa": "Varon 55a solicita receta de medicacion habitual porque se le acabo. Niega sintomas. TA 130/76, FC 74, FR 16, SatO2 98%, T 36.5, Glasgow 15."},
    {"id": "S20", "categoria": "administrativo", "motivo": "Solicita justificante medico para el trabajo.", "edad_sexo": "Mujer 28a solicita justificante medico para el trabajo.", "motivo_constantes": "Mujer 28a solicita justificante medico. TA 116/70, FC 72, FR 15, SatO2 99%, T 36.6.", "motivo_antecedentes": "Mujer 28a solicita justificante medico para el trabajo. Niega sintomas, dolor y fiebre.", "completa": "Mujer 28a solicita justificante medico para el trabajo. Niega sintomas, dolor y fiebre. TA 116/70, FC 72, FR 15, SatO2 99%, T 36.6."},
]

VARIANTES = ["motivo", "edad_sexo", "motivo_constantes", "motivo_antecedentes", "completa"]
pd.DataFrame(CASOS)[["id", "categoria", "completa"]]

,id,categoria,completa
0,S01,critico_neuro,Varon 67a. HTA y FA cronica. Tto bisoprolol y ...
1,S02,dolor_toracico,Mujer 72a con diabetes e hipertension. Dolor t...
2,S03,disnea,Varon 64a EPOC. Consulta por disnea progresiva...
3,S04,sepsis,Mujer 81a con insuficiencia renal. Fiebre y de...
4,S05,cefalea,"Hombre de unos 40 anos con cefalea brusca, la ..."
5,S06,abdomen,Mujer 52a dolor hipocondrio derecho desde la c...
6,S07,abdomen_fid,Varon 29a dolor en fosa iliaca derecha de 10 h...
7,S08,sincope,Mujer 58a sincope en domicilio con recuperacio...
8,S09,trauma,"Varon 35a caida de bicicleta hace 1h, dolor in..."
9,S10,alergia,Mujer 24a urticaria generalizada tras comer fr...


## 3. Ejecucion de ablaciones

Cada variante pasa por el extractor LLM y por el mismo predictor final. Se guardan errores para revision manual si una variante resulta demasiado incompleta para el esquema.

In [3]:
def contar_constantes(vector) -> int:
    campos = [
        vector.presion_sistolica,
        vector.presion_diastolica,
        vector.frecuencia_cardiaca,
        vector.frecuencia_respiratoria,
        vector.saturacion_oxigeno,
        vector.temperatura,
    ]
    return sum(c is not None for c in campos)

def evaluar_narrativa(texto: str) -> dict:
    t0 = time.perf_counter()
    vector = extraer_vector_clinico(texto, modelo=LLM_MODEL)
    alertas = validar_vector_clinico(vector, texto)
    resultado = predictor.predict(vector, texto)
    return {
        "vector": vector,
        "alertas": alertas,
        "resultado": resultado,
        "latencia_s": time.perf_counter() - t0,
    }

filas = []
cache = {}

for caso in CASOS:
    referencia = None
    for variante in VARIANTES:
        texto = caso[variante]
        clave = (caso["id"], variante)
        try:
            cache[clave] = evaluar_narrativa(texto)
            obj = cache[clave]
            vector = obj["vector"]
            resultado = obj["resultado"]
            if variante == "completa":
                referencia = resultado
            filas.append({
                "id": caso["id"],
                "categoria": caso["categoria"],
                "variante": variante,
                "n_caracteres": len(texto),
                "n_sintomas": len(vector.sintomas_presentes),
                "sintomas": ", ".join(vector.sintomas_presentes),
                "n_constantes": contar_constantes(vector),
                "n_antecedentes": len(vector.patologias_previas),
                "n_medicacion": len(vector.medicacion_habitual),
                "n_alertas": len(obj["alertas"]),
                "alertas": resumen_validacion(obj["alertas"]),
                "esi_predicho": resultado.clase_predicha,
                "confianza": resultado.confianza,
                "p_esi1": float(resultado.probas[0]),
                "alerta_a1": resultado.alerta_a1_activada,
                "latencia_s": obj["latencia_s"],
                "error": "",
            })
        except Exception as exc:
            filas.append({
                "id": caso["id"],
                "categoria": caso["categoria"],
                "variante": variante,
                "n_caracteres": len(texto),
                "error": repr(exc),
            })

ablacion = pd.DataFrame(filas)
ablacion

2026-06-04 18:07:23.491 | INFO     | triaje_ia.llm.extractor:extraer_vector_clinico:147 - Iniciando extracción con modelo 'llama3.1:8b-instruct-q4_K_M'
2026-06-04 18:07:48.846 | SUCCESS  | triaje_ia.llm.extractor:extraer_vector_clinico:163 - Extraídos 1 síntomas
2026-06-04 18:07:48.855 | INFO     | triaje_ia.data.features:_calcular_missingness_vitales:176 - Missingness vitales calculado
2026-06-04 18:07:48.859 | INFO     | triaje_ia.data.features:_calcular_banderas_vitales:197 - Banderas vitales calculadas
2026-06-04 18:07:48.866 | INFO     | triaje_ia.data.features:_calcular_scores_compuestos:305 - Scores compuestos calculados
2026-06-04 18:07:48.868 | INFO     | triaje_ia.data.features:calcular_features_bloque1:315 - Bloque 1 completado: missingness + banderas + scores
2026-06-04 18:07:48.872 | INFO     | triaje_ia.data.features:calcular_features_bloque2:345 - Bloque 2 completado: demográfico y logístico
2026-06-04 18:07:48.897 | INFO     | triaje_ia.data.features:calcular_features_b

,id,categoria,variante,n_caracteres,n_sintomas,sintomas,n_constantes,n_antecedentes,n_medicacion,n_alertas,alertas,esi_predicho,confianza,p_esi1,alerta_a1,latencia_s,error
0,S01,critico_neuro,motivo,41,1,altered mental status,0,0,0,0,Sin alertas de validacion,1,0.867468,0.867468,False,41.638209,
1,S01,critico_neuro,edad_sexo,52,1,altered mental status,0,0,0,0,Sin alertas de validacion,1,0.904162,0.904162,False,8.541877,
2,S01,critico_neuro,motivo_constantes,107,1,altered mental status,6,0,0,0,Sin alertas de validacion,1,0.625538,0.625538,False,9.497478,
3,S01,critico_neuro,motivo_antecedentes,101,1,altered mental status,0,2,2,0,Sin alertas de validacion,1,0.903059,0.903059,False,9.655101,
4,S01,critico_neuro,completa,199,2,"altered mental status, bradycardia",6,2,2,0,Sin alertas de validacion,1,0.508077,0.508077,False,10.075047,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,S20,administrativo,motivo,45,1,administrative request,0,0,0,0,Sin alertas de validacion,1,0.625725,0.625725,False,7.025720,
96,S20,administrativo,edad_sexo,55,1,administrative request,0,0,0,0,Sin alertas de validacion,1,0.629087,0.629087,False,7.595632,
97,S20,administrativo,motivo_constantes,83,1,administrative request,6,0,0,0,Sin alertas de validacion,4,0.359625,0.091488,False,8.302575,
98,S20,administrativo,motivo_antecedentes,87,1,administrative request,0,0,0,2,"Validación: 0 errores, 1 warnings\n INFO [niv...",1,0.629087,0.629087,False,6.635002,


## 4. Comparacion contra la historia completa

La estabilidad se calcula respecto a la variante completa de cada caso.

In [4]:
referencias = ablacion[ablacion["variante"] == "completa"][[
    "id", "esi_predicho", "confianza", "p_esi1", "alerta_a1"
]].rename(columns={
    "esi_predicho": "esi_completa",
    "confianza": "confianza_completa",
    "p_esi1": "p_esi1_completa",
    "alerta_a1": "alerta_a1_completa",
})

comparacion = ablacion.merge(referencias, on="id", how="left")
comparacion["misma_clase_que_completa"] = comparacion["esi_predicho"] == comparacion["esi_completa"]
comparacion["delta_p_esi1"] = comparacion["p_esi1"] - comparacion["p_esi1_completa"]
comparacion["delta_confianza"] = comparacion["confianza"] - comparacion["confianza_completa"]
comparacion[[
    "id", "categoria", "variante", "n_caracteres", "n_constantes", "n_alertas",
    "esi_predicho", "esi_completa", "misma_clase_que_completa",
    "p_esi1", "p_esi1_completa", "delta_p_esi1", "alerta_a1", "alerta_a1_completa", "error",
]]

,id,categoria,variante,n_caracteres,n_constantes,n_alertas,esi_predicho,esi_completa,misma_clase_que_completa,p_esi1,p_esi1_completa,delta_p_esi1,alerta_a1,alerta_a1_completa,error
0,S01,critico_neuro,motivo,41,0,0,1,1,True,0.867468,0.508077,0.359392,False,False,
1,S01,critico_neuro,edad_sexo,52,0,0,1,1,True,0.904162,0.508077,0.396085,False,False,
2,S01,critico_neuro,motivo_constantes,107,6,0,1,1,True,0.625538,0.508077,0.117462,False,False,
3,S01,critico_neuro,motivo_antecedentes,101,0,0,1,1,True,0.903059,0.508077,0.394983,False,False,
4,S01,critico_neuro,completa,199,6,0,1,1,True,0.508077,0.508077,0.000000,False,False,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,S20,administrativo,motivo,45,0,0,1,4,False,0.625725,0.091488,0.534237,False,False,
96,S20,administrativo,edad_sexo,55,0,0,1,4,False,0.629087,0.091488,0.537598,False,False,
97,S20,administrativo,motivo_constantes,83,6,0,4,4,True,0.091488,0.091488,0.000000,False,False,
98,S20,administrativo,motivo_antecedentes,87,0,2,1,4,False,0.629087,0.091488,0.537598,False,False,


## 5. Resumen por nivel de informacion

Esta tabla resume que variantes mantienen mejor la prediccion de la historia completa. Debe interpretarse como sensibilidad tecnica del prototipo, no como validacion clinica.

In [5]:
resumen_variantes = (
    comparacion.groupby("variante", dropna=False)
    .agg(
        n=("id", "count"),
        errores=("error", lambda s: int((s.fillna("") != "").sum())),
        caracteres_medios=("n_caracteres", "mean"),
        constantes_medias=("n_constantes", "mean"),
        alertas_medias=("n_alertas", "mean"),
        misma_clase_pct=("misma_clase_que_completa", "mean"),
        delta_p_esi1_abs_medio=("delta_p_esi1", lambda s: float(np.nanmean(np.abs(s)))),
        delta_confianza_abs_medio=("delta_confianza", lambda s: float(np.nanmean(np.abs(s)))),
    )
    .reset_index()
)
resumen_variantes

,variante,n,errores,caracteres_medios,constantes_medias,alertas_medias,misma_clase_pct,delta_p_esi1_abs_medio,delta_confianza_abs_medio
0,completa,20,0,139.00,6.0,0.15,1.00,0.000000,0.000000
1,edad_sexo,20,0,60.35,0.0,0.45,0.35,0.492988,0.211690
2,motivo,20,0,45.55,0.0,0.45,0.35,0.477680,0.216996
3,motivo_antecedentes,20,0,85.80,0.0,0.60,0.35,0.486219,0.214264
4,motivo_constantes,20,0,92.55,6.0,0.10,0.95,0.013685,0.040634


## 6. Casos que cambian de nivel ESI

Estos casos deben revisarse manualmente porque muestran sensibilidad a informacion omitida.

In [6]:
cambios = comparacion[
    (comparacion["variante"] != "completa")
    & (comparacion["error"].fillna("") == "")
    & (~comparacion["misma_clase_que_completa"])
].copy()

cambios[[
    "id", "categoria", "variante", "sintomas", "n_constantes", "n_alertas",
    "esi_predicho", "esi_completa", "p_esi1", "p_esi1_completa", "delta_p_esi1",
]]

,id,categoria,variante,sintomas,n_constantes,n_alertas,esi_predicho,esi_completa,p_esi1,p_esi1_completa,delta_p_esi1
20,S05,cefalea,motivo,"thunderclap headache, vomiting",0,0,1,3,0.923768,0.144271,0.779497
21,S05,cefalea,edad_sexo,thunderclap headache,0,0,1,3,0.856466,0.144271,0.712195
23,S05,cefalea,motivo_antecedentes,"thunderclap headache, vomiting, photophobia",0,0,1,3,0.870647,0.144271,0.726376
25,S06,abdomen,motivo,"right upper quadrant abdominal pain, nausea",0,1,1,3,0.830008,0.020565,0.809443
26,S06,abdomen,edad_sexo,"right upper quadrant abdominal pain, nausea",0,1,1,3,0.859441,0.020565,0.838876
28,S06,abdomen,motivo_antecedentes,"right upper quadrant abdominal pain, nausea",0,1,1,3,0.859441,0.020565,0.838876
30,S07,abdomen_fid,motivo,"right lower quadrant abdominal pain, vomiting",0,1,1,3,0.888450,0.026821,0.861628
31,S07,abdomen_fid,edad_sexo,"right lower quadrant abdominal pain, vomiting",0,1,1,3,0.893034,0.026821,0.866213
33,S07,abdomen_fid,motivo_antecedentes,"right lower quadrant abdominal pain, vomiting",0,0,1,3,0.656766,0.026821,0.629945
35,S08,sincope,motivo,syncope,0,0,1,2,0.706925,0.205539,0.501385


## 7. Guia minima propuesta para enfermeria

A partir de esta auditoria, la informacion minima recomendable para estabilizar el flujo es:

1. Edad y sexo del paciente.
2. Motivo principal de consulta, con localizacion anatomica si hay dolor.
3. Tiempo de evolucion o inicio aproximado.
4. Constantes vitales: TA, FC, FR, SatO2 y temperatura.
5. Dolor en escala 0-10 cuando proceda.
6. Antecedentes relevantes para urgencias: cardiacos, respiratorios, neurologicos, renales, metabolicos, anticoagulacion, epilepsia, embarazo si aplica.
7. Medicacion critica o habitual relevante, especialmente anticoagulantes, betabloqueantes, insulina, antiagregantes, diureticos y psicofarmacos.
8. Signos de alarma expresados de forma literal: confusion, bajo nivel de conciencia, disnea, dolor toracico, sincope, sangrado, fiebre alta, hipotension, hipoxemia, estridor o convulsion.

Esta guia no sustituye el juicio clinico. Su funcion es mejorar la calidad de la entrada textual para que el extractor no tenga que inferir datos ausentes.

## 8. Resultados principales, conclusiones y campos imprescindibles

La auditoria se realizo sobre 20 casos sinteticos representativos de distintos escenarios de urgencias y cinco niveles de informacion por caso: solo motivo de consulta, motivo con edad/sexo, motivo con constantes, motivo con antecedentes y narrativa completa. La narrativa completa se uso como referencia interna de comparacion para medir la sensibilidad del flujo LLM -> adaptador -> modelo ante entradas progresivamente incompletas.

Los resultados muestran una diferencia clara entre las variantes con y sin constantes vitales. La variante `motivo_constantes`, que incluye motivo de consulta, edad/sexo y constantes vitales, mantuvo la misma clase ESI que la narrativa completa en el 95% de los casos, con una variacion media absoluta de `P(ESI 1)` de 0,0125 y una variacion media de confianza de 0,0404. En cambio, las variantes sin constantes vitales mantuvieron la misma clase solo en el 35% de los casos y mostraron variaciones medias absolutas de `P(ESI 1)` cercanas a 0,48-0,49. Esto indica que el sistema es muy sensible a la ausencia de constantes y tiende a generar estimaciones mucho menos estables cuando solo recibe texto sindromico o antecedentes.

La revision de los casos que cambiaron de nivel ESI confirma este patron. La mayoria de discrepancias aparecieron en variantes sin constantes, especialmente en cuadros como cefalea brusca, dolor abdominal, sincope, trauma, rectorragia, convulsion, fiebre leve y motivos administrativos. En esas versiones incompletas, el modelo tendio con frecuencia a desplazar la prediccion hacia ESI 1 y a aumentar artificialmente `P(ESI 1)`. La unica discrepancia observada en la variante `motivo_constantes` fue el caso `S15` de otalgia, donde la prediccion paso de ESI 3 en la narrativa completa a ESI 4 en la version con constantes. En ese caso, la diferencia en `P(ESI 1)` fue pequena, por lo que el cambio refleja mas una diferencia de estratificacion entre baja y media prioridad que un problema de deteccion de criticidad.

A partir de esta auditoria, los campos imprescindibles para una entrada minima estable son:

| Bloque | Campo imprescindible | Justificacion operativa |
|---|---|---|
| Identificacion clinica basica | Edad y sexo | Permiten contextualizar riesgo, fragilidad y umbrales fisiologicos. |
| Motivo de consulta | Problema principal expresado de forma literal | Activa las variables regex de `chiefcomplaint` y la representacion semantica Bio_ClinicalBERT/SVD. |
| Temporalidad | Inicio o tiempo de evolucion | Ayuda a distinguir cuadros bruscos, progresivos, persistentes o banales. |
| Constantes vitales | TA sistolica/diastolica, FC, FR, SatO2 y temperatura | Fue el bloque que mas estabilizo la prediccion respecto a la narrativa completa. |
| Dolor | Escala 0-10 cuando proceda | Es una variable directa del modelo y mejora la interpretacion de cuadros dolorosos. |
| Signos de alarma | Bajo nivel de conciencia, confusion, disnea, dolor toracico, sincope, sangrado, convulsion, hipoxemia, hipotension o fiebre alta | Evita que el extractor tenga que inferir gravedad a partir de texto incompleto. |
| Antecedentes relevantes | Cardiopatia, EPOC/asma, enfermedad renal, diabetes, epilepsia, anticoagulacion, embarazo u otros antecedentes clinicamente pertinentes | Aportan contexto, aunque no sustituyen a las constantes vitales. |
| Medicacion relevante | Anticoagulantes, antiagregantes, betabloqueantes, insulina, diureticos, opioides y psicofarmacos | Permite activar proxies de comorbilidad y riesgo farmacologico. |

Como conclusion, el minimo operativo no debe limitarse al motivo de consulta. Para este prototipo, una entrada textual suficientemente estable debe incluir al menos motivo, edad/sexo y constantes vitales completas. Los antecedentes y la medicacion mejoran el contexto clinico, pero en esta auditoria no compensaron la ausencia de constantes. Esta seccion debe interpretarse como una auditoria tecnica de sensibilidad del prototipo, no como una validacion clinica ni como una guia asistencial cerrada.
